# 🎙️ Textless Tamil Speech-to-Speech AI (V2) — Google Colab Training
### Milestone A: Continuous SSL Feature Mapping to Discrete Neural Audio Codecs

This notebook trains a **100% textless Tamil Speech-to-Speech model** directly on continuous streaming audio from Hugging Face (`SPRINGLab/IndicVoices-R_Tamil`).

---
### Strict Architectural Guarantees:
- **Zero Text Conditioning:** No transcripts, no phoneme sequences, no tokenizers, no text encoders.
- **Continuous SSL Input:** Meta MMS-300M representations (16 kHz, 50 Hz, 1024-dim).
- **Discrete RVQ Output:** Encodec 24 kHz acoustic tokens (75 Hz, 4 codebooks).
- **Continuous Timestamp Alignment:** Exact mathematical temporal interpolation aligning 20.0 ms MMS frames to 13.33 ms Encodec frames.
- **Two-Stage Codebook Strategy:**
  - **Stage 1:** Focus 100% loss on Codebook 0 (the phonetic backbone) to reach high Top-1 accuracy and eliminate repetition attractors.
  - **Stage 2:** Train depth-conditioned heads for residual codebooks CB1–CB3.

## Step 1: Check GPU Acceleration
Ensure that a GPU runtime is selected (**Runtime > Change runtime type > T4 / V100 / A100**).

In [ ]:
!nvidia-smi

## Step 2: Clone Repository & Checkout Milestone A Branch

In [ ]:
!git clone https://github.com/sathiyaseelank-dot/tamil-voice-ai.git
%cd tamil-voice-ai
!git checkout feat/s2s-v2-milestone-a
!git status

## Step 3: Install Required Dependencies

In [ ]:
!pip install -q soundfile librosa encodec transformers datasets accelerate torchaudio einops

## Step 4: Mount Google Drive (Recommended for Persistent Checkpoints)
Mounting Google Drive ensures that checkpoints (`.pt`) and decoded audio samples (`.wav`) are preserved permanently across Colab sessions.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/tamil_voice_ai_checkpoints/s2s_v2'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"✓ Google Drive mounted! Checkpoints will persist in: {SAVE_DIR}")
except Exception as e:
    SAVE_DIR = 'checkpoints_s2s_v2'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Running without Drive mount. Saving locally to: {SAVE_DIR}")

## Step 5: Test Streaming Tamil Speech Dataset
Preview real Tamil speech streaming from Hugging Face (`SPRINGLab/IndicVoices-R_Tamil`).

In [ ]:
import IPython.display as ipd
from datasets import load_dataset, Audio

print("Connecting to streaming IndicVoices-R Tamil dataset...")
dataset = load_dataset("SPRINGLab/IndicVoices-R_Tamil", split="train", streaming=True)
dataset = dataset.cast_column("audio", Audio(sampling_rate=24000))

sample = next(iter(dataset))
audio_data = sample["audio"]["array"]
print(f"✓ Audio stream active! Sample length: {len(audio_data)/24000.0:.2f} seconds")
ipd.Audio(audio_data, rate=24000)

## Step 6: Launch Milestone A Stage 1 Training (CB0 Phonetic Backbone)
In Stage 1, we dedicate 100% of the loss to Codebook 0 (the phonetic envelope). Residual codebooks are disabled during backpropagation to prevent phase noise from corrupting early learning.

Key parameters:
- `--stage 1`: 100% loss on CB0
- `--init_type mms`: Initializes the 6-layer Transformer from Meta MMS-300M pretrained encoder
- `--max_steps 5000`: 5,000 steps (~1-2 hours on T4 GPU)

In [ ]:
!python train_s2s_v2_streaming.py \
    --stage 1 \
    --init_type mms \
    --max_steps 5000 \
    --lr 2e-4 \
    --eval_interval 250 \
    --save_interval 500 \
    --save_dir "$SAVE_DIR"

## Step 7: Listen to Decoded Speech Samples
Compare the CB0-only decoded audio against the ground truth reference.

In [ ]:
import glob
import IPython.display as ipd

samples_dir = os.path.join(SAVE_DIR, "samples")
cb0_samples = sorted(glob.glob(f"{samples_dir}/*cb0_only.wav"))

if cb0_samples:
    latest_cb0 = cb0_samples[-1]
    print(f"Playing latest CB0-decoded audio: {latest_cb0}")
    ipd.display(ipd.Audio(latest_cb0, rate=24000))
else:
    print("No sample files found yet in:", samples_dir)

## Step 8: (Optional) Launch Milestone A Stage 2 Training (Residual Harmonics)
Once Stage 1 reaches >60% CB0 accuracy, run Stage 2 to train the DepthAudioHead on residual codebooks (CB1–CB3).

In [ ]:
# Resume from Stage 1 checkpoint and train residual codebooks
latest_stage1_ckpt = os.path.join(SAVE_DIR, "s2s_v2_stage1_latest.pt")

!python train_s2s_v2_streaming.py \
    --stage 2 \
    --resume "$latest_stage1_ckpt" \
    --max_steps 10000 \
    --lr 1e-4 \
    --eval_interval 250 \
    --save_interval 500 \
    --save_dir "$SAVE_DIR"